In [2]:
##Menyiapkan Dataset
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [8]:
##Membuat Sparksession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("Tugas7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Siap untuk mengerjakan tugas")

Siap untuk mengerjakan tugas


In [4]:
##A.EXTRACT
#Membaca file json dan csv dari HDFS
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
                        header=True, inferSchema=True)
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
                        header=True, inferSchema=True)

#Memanggil dataframe yang sudah dibuat
print("dataframe cabang")
df_cabang.show(5)
print("dataframe menu")
df_menu.show(5)
print("dataframe transaksi")
df_transaksi.show(5)

dataframe cabang
+---------+--------------+----------+--------------------+
|cabang_id|kepala_barista|      kota|         nama_cabang|
+---------+--------------+----------+--------------------+
|        1|         Nadia|  Magelang|Kopi Nusantara Ma...|
|        2|          Reza|Yogyakarta|Kopi Nusantara Yo...|
|        3|         Nadia|  Semarang|Kopi Nusantara Se...|
+---------+--------------+----------+--------------------+

dataframe menu
+-------+---------+--------------+-----+
|menu_id|nama_menu| kategori_menu|harga|
+-------+---------+--------------+-----+
|      1|   Menu-1|Makanan Ringan|35000|
|      2|   Menu-2|Makanan Ringan|22000|
|      3|   Menu-3|Makanan Ringan|30000|
|      4|   Menu-4|      Non-Kopi|22000|
|      5|   Menu-5|          Kopi|22000|
+-------+---------+--------------+-----+
only showing top 5 rows

dataframe transaksi
+------+---------+-------+---+
|trx_id|cabang_id|menu_id|qty|
+------+---------+-------+---+
|   KN0|        1|     14|  4|
|   KN1|        

In [5]:
##B.Transform
#Menggabungkan ketiga dataframe
df_gabung = df_transaksi.join(df_menu, on="menu_id", how="left")\
    .join(df_cabang, on="cabang_id", how="left")

#Menambahkan kolom ke total_penjualan kedalam dataframe gabungan dan memanggil df
df_gabung = df_gabung.withColumn("total_penjualan",col("qty")*col("harga"))
df_gabung.show(5)

+---------+-------+------+---+---------+--------------+-----+--------------+----------+--------------------+---------------+
|cabang_id|menu_id|trx_id|qty|nama_menu| kategori_menu|harga|kepala_barista|      kota|         nama_cabang|total_penjualan|
+---------+-------+------+---+---------+--------------+-----+--------------+----------+--------------------+---------------+
|        1|     14|   KN0|  4|  Menu-14|          Kopi|18000|         Nadia|  Magelang|Kopi Nusantara Ma...|          72000|
|        1|     11|   KN1|  4|  Menu-11|          Kopi|22000|         Nadia|  Magelang|Kopi Nusantara Ma...|          88000|
|        1|     11|   KN2|  3|  Menu-11|          Kopi|22000|         Nadia|  Magelang|Kopi Nusantara Ma...|          66000|
|        3|     11|   KN3|  1|  Menu-11|          Kopi|22000|         Nadia|  Semarang|Kopi Nusantara Se...|          22000|
|        2|     12|   KN4|  4|  Menu-12|Makanan Ringan|35000|          Reza|Yogyakarta|Kopi Nusantara Yo...|         140000|


In [6]:
##C.TRANSFORM
#Membuat ringkasan total penjualan berdasarkan cabang, nama menu dan kategori menu
ringkasan = df_gabung.groupBy("cabang_id", "nama_menu","kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)
#Membuat partisi berdasarkan cabang diurutkan dari total penjualan terbesar
window_rank= Window.partitionBy("cabang_id").orderBy(col("total_penjualan").desc())

#Membuat kolom rank untuk memberi peringkat 
df_ranked = ringkasan.withColumn("rank", row_number().over(window_rank))

#Memfilter dataframe untuk memunculkan top 2 diurutkan dari nama dan rank
df_ranked=df_ranked.filter(col("rank")<=2)
                              
df_ranked.show()

+---------+---------+--------------+---------------+----+
|cabang_id|nama_menu| kategori_menu|total_penjualan|rank|
+---------+---------+--------------+---------------+----+
|        1|   Menu-1|Makanan Ringan|        8505000|   1|
|        1|  Menu-12|Makanan Ringan|        8085000|   2|
|        2|  Menu-12|Makanan Ringan|        9275000|   1|
|        2|  Menu-15|      Non-Kopi|        8160000|   2|
|        3|  Menu-13|Makanan Ringan|        8750000|   1|
|        3|   Menu-9|Makanan Ringan|        8715000|   2|
+---------+---------+--------------+---------------+----+



In [66]:
#D.TRANSFORM
#Menampilkan total penjualan tiap kepala barista diurutkan dari yang tertinggi
df_gabung.createOrReplaceTempView("ranked")
df_barista=spark.sql('''
select kepala_barista, sum(total_penjualan) as total_penjualan
from ranked
group by kepala_barista
order by total_penjualan desc
''')
df_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [67]:
#E.LOAD
#Membuat direktori baru bernama processed
!hdfs dfs -mkdir /user/mahasiswa/tugas7/processed/

#Menyimpan df_gabung menjadi laporan di hdfs dan dipartisi berdasarkan kategori menu
df_gabung.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)
#Mengecek apakah file sudah ada didalam direktori
!hdfs dfs -ls -R /user/mahasiswa/tugas7/processed/laporan


-rw-r--r--   3 celloz supergroup          0 2026-09-25 20:18 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - celloz supergroup          0 2026-09-25 20:18 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
-rw-r--r--   3 celloz supergroup      12309 2026-09-25 20:18 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi/part-00000-bc73f86f-6dc7-4876-8863-07eb80ef31d1.c000.snappy.parquet
drwxr-xr-x   - celloz supergroup          0 2026-09-25 20:18 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
-rw-r--r--   3 celloz supergroup      16393 2026-09-25 20:18 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan/part-00000-bc73f86f-6dc7-4876-8863-07eb80ef31d1.c000.snappy.parquet
drwxr-xr-x   - celloz supergroup          0 2026-09-25 20:18 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi
-rw-r--r--   3 celloz supergroup       8552 2026-09-25 20:18 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi

In [10]:
#Mengecek jumlah baris apakah sama seperti sebelum disimpan
df_verif= spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")
print("Jumlah baris dari parquet yang disimpan baru saja ialah: ",df_verif.count())

Jumlah baris dari parquet yang disimpan baru saja ialah:  4000


In [15]:
#F. Ringkasan Eksekutif
#Menampilkan total penjualan per cabang, jumlah transaksinya, serta rata-rata nilai transaksi
df_verif.createOrReplaceTempView("verif")
df_cek=spark.sql('''
select cabang_id, nama_cabang, kota, kepala_barista, sum(total_penjualan) as total_pendapatan,
count(trx_id) as jumlah_transaksi, avg(total_penjualan) as rata_rata_nilai_transaksi
from verif
group by cabang_id, nama_cabang, kota, kepala_barista
order by total_pendapatan desc
''')
df_cek.show()

+---------+--------------------+----------+--------------+----------------+----------------+-------------------------+
|cabang_id|         nama_cabang|      kota|kepala_barista|total_pendapatan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------+--------------------+----------+--------------+----------------+----------------+-------------------------+
|        3|Kopi Nusantara Se...|  Semarang|         Nadia|        90542000|            1393|         64997.8463747308|
|        1|Kopi Nusantara Ma...|  Magelang|         Nadia|        87138000|            1302|        66926.26728110599|
|        2|Kopi Nusantara Yo...|Yogyakarta|          Reza|        84645000|            1305|        64862.06896551724|
+---------+--------------------+----------+--------------+----------------+----------------+-------------------------+



In [7]:
spark.stop()
print("tugasmu sudah selesai spark")

tugasmu sudah selesai spark
